In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# BASE_DIR = DIR of the dataset
BASE_DIR = "/kaggle/input/competitions/ima205-challenge-2026/IMA205-challenge"

# WEIGHTS_DIR = DIR of weights from 5models_k_fold.ipynb
WEIGHTS_DIR = "/kaggle/input/models/mohamedyassinekhales/otherweights/other/default/1" 

train_df = pd.read_csv(f"{BASE_DIR}/train_metadata.csv")
test_df = pd.read_csv(f"{BASE_DIR}/test_metadata.csv")

L = train_df['label'].unique()
dict_labels = {L[i]: i for i in range(len(L))}
inv_dict_labels = {v: k for k, v in dict_labels.items()}

val_test_transforms = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class BloodCellTestDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df
        self.root_dir = root_dir
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, self.df.iloc[idx]['ID'])
        image = Image.open(img_path).convert('RGB')
        return self.transform(image) if self.transform else image

test_data = BloodCellTestDataset(test_df, f"{BASE_DIR}/test", transform=val_test_transforms)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

# Matrix containing the test probabilities of each class for a given image
total_probs = np.zeros((len(test_df), len(L)))

# models importance multipliers
model_weights = {
    "densenet": 1.5,       # Best model (0.7983)
    "efficientnet_v2": 1.3, # Runner Up (0.7938)
    "resnet": 1.1,         # Solid (0.7893)
    "convnext": 1.0,       # Good (0.7871)
    "efficientnet_b3": 1.0 # Good (0.78)
}

fold_files = [f for f in os.listdir(WEIGHTS_DIR) if f.endswith('.pth')]

for file in fold_files:
    print(f"\nLoading {file}...")
    
    # 1. Dynamically load the correct architecture based on the filename
    file_lower = file.lower()
    
    if "Best_Model" in file or "efficientnet_b3" in file:
        model = models.efficientnet_b3(weights=None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(L))
        current_weight = model_weights["efficientnet_b3"]
        
    elif "resnet" in file:
        model = models.resnet50(weights=None)
        model.fc = nn.Linear(model.fc.in_features, len(L))
        current_weight = model_weights["resnet"]
        
    elif "convnet" in file or "convnext" in file:
        model = models.convnext_tiny(weights=None)
        model.classifier[2] = nn.Linear(model.classifier[2].in_features, len(L))
        current_weight = model_weights["convnext"]
        
    elif "efficientnet_v2" in file:
        model = models.efficientnet_v2_s(weights=None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(L))
        current_weight = model_weights["efficientnet_v2"]
        
    elif "densenet" in file:
        model = models.densenet121(weights=None)
        model.classifier = nn.Linear(model.classifier.in_features, len(L))
        current_weight = model_weights["densenet"]
        

    # Load weights
    model.load_state_dict(torch.load(os.path.join(WEIGHTS_DIR, file), map_location=device))
    model = model.to(device)
    model.eval()
    
    fold_probs = []
    with torch.no_grad():
        for images in tqdm(test_loader, desc=f"TTA Inference (Weight: {current_weight}x)"):
            images = images.to(device)
            
            # Test-Time Augmentation (TTA)
            img_hflip = torch.flip(images, dims=[3])       
            img_vflip = torch.flip(images, dims=[2])       
            img_rot = torch.rot90(images, k=1, dims=[2, 3]) 
            
            with torch.amp.autocast('cuda'):
                p_orig = F.softmax(model(images), dim=1)
                p_hflip = F.softmax(model(img_hflip), dim=1)
                p_vflip = F.softmax(model(img_vflip), dim=1)
                p_rot = F.softmax(model(img_rot), dim=1)
            
            # Average the 4 angles
            prob_avg = (p_orig + p_hflip + p_vflip + p_rot) / 4.0
            
            # MULTIPLY BY THIS MODEL'S VOTING POWER
            weighted_prob = prob_avg * current_weight 
            
            fold_probs.extend(weighted_prob.cpu().numpy())
            
    total_probs += np.array(fold_probs)

final_predictions = np.argmax(total_probs, axis=1)

test_df['label'] = [inv_dict_labels[p] for p in final_predictions]
submission_file = 'submission_Weighted_Ensemble.csv'
test_df.to_csv(submission_file, index=False)
print(f"Submission saved.")